<a href="https://colab.research.google.com/github/StarShoppingUG/CropWise/blob/master/FakeNewsDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import os

os.listdir('/content/drive/MyDrive/ML_Project')

['Fake.csv', 'True.csv']

In [ ]:
import pandas as pd

fake_path = "/content/drive/MyDrive/ML_Project/Fake.csv"
true_path = "/content/drive/MyDrive/ML_Project/True.csv"

fake_df = pd.read_csv(fake_path)
true_df = pd.read_csv(true_path)

print(fake_df.head())
print(true_df.head())

                                               title  ...               date
0   Donald Trump Sends Out Embarrassing New Year’...  ...  December 31, 2017
1   Drunk Bragging Trump Staffer Started Russian ...  ...  December 31, 2017
2   Sheriff David Clarke Becomes An Internet Joke...  ...  December 30, 2017
3   Trump Is So Obsessed He Even Has Obama’s Name...  ...  December 29, 2017
4   Pope Francis Just Called Out Donald Trump Dur...  ...  December 25, 2017

[5 rows x 4 columns]
                                               title  ...                date
0  As U.S. budget fight looms, Republicans flip t...  ...  December 31, 2017 
1  U.S. military to accept transgender recruits o...  ...  December 29, 2017 
2  Senior U.S. Republican senator: 'Let Mr. Muell...  ...  December 31, 2017 
3  FBI Russia probe helped by Australian diplomat...  ...  December 30, 2017 
4  Trump wants Postal Service to charge 'much mor...  ...  December 29, 2017 

[5 rows x 4 columns]


In [ ]:
fake_df["label"] = 0
true_df["label"] = 1

In [ ]:
df = pd.concat([fake_df, true_df], axis=0)

df = df.reset_index(drop=True)

print(df.shape)
df.head()

(44898, 5)


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [ ]:
df["content"] = df["title"] + " " + df["text"]

In [ ]:
import re
import string

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>+', '', text)
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text)
    text = re.sub(r'\n', '', text)
    return text

df["content"] = df["content"].apply(clean_text)

In [ ]:
from sklearn.model_selection import train_test_split

X = df["content"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(stop_words="english", max_df=0.7)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X_train_vec, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

y_pred_lr = model_lr.predict(X_test_vec)

print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Accuracy: 0.9844097995545658
              precision    recall  f1-score   support

           0       0.99      0.98      0.99      4733
           1       0.98      0.98      0.98      4247

    accuracy                           0.98      8980
   macro avg       0.98      0.98      0.98      8980
weighted avg       0.98      0.98      0.98      8980



In [ ]:
def predict_news(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])
    pred = model_lr.predict(vec)

    return "Real News" if pred[0] == 1 else "Fake News"

predict_news("Government announces new education reforms to improve schools")

'Real News'

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_nb = MultinomialNB()
model_nb.fit(X_train_vec, y_train)

MultinomialNB()

In [ ]:

y_pred_nb = model_nb.predict(X_test_vec)

print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.9364142538975501
              precision    recall  f1-score   support

           0       0.94      0.94      0.94      4733
           1       0.94      0.93      0.93      4247

    accuracy                           0.94      8980
   macro avg       0.94      0.94      0.94      8980
weighted avg       0.94      0.94      0.94      8980



In [ ]:
print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_nb))

Logistic Regression Accuracy: 0.9844097995545658
Naive Bayes Accuracy: 0.9364142538975501


In [ ]:

import numpy as np

def predict_news(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])

    # Logistic Regression prediction
    pred = model_lr.predict(vec)[0]

    # Confidence score
    prob = model_lr.predict_proba(vec)[0]
    confidence = np.max(prob) * 100

    if pred == 1:
        return f"🟢 REAL NEWS ({confidence:.2f}% confidence)"
    else:
        return f"🔴 FAKE NEWS ({confidence:.2f}% confidence)"

In [ ]:
def compare_news(text):
    text = clean_text(text)
    vec = vectorizer.transform([text])

    lr_pred = model_lr.predict(vec)[0]
    nb_pred = model_nb.predict(vec)[0]

    result = ""

    result += f"Logistic Regression: {'REAL' if lr_pred==1 else 'FAKE'}\n"
    result += f"Naive Bayes: {'REAL' if nb_pred==1 else 'FAKE'}"

    return result

In [ ]:

import gradio as gr

with gr.Blocks() as interface:

    gr.Markdown("# 📰 Fake News Detection System")

    with gr.Tab("Single Prediction"):
        input_text = gr.Textbox(lines=6, label="Enter News Article")
        output = gr.Textbox(label="Prediction")
        btn = gr.Button("Predict")
        btn.click(fn=predict_news, inputs=input_text, outputs=output)

    with gr.Tab("Model Comparison"):
        input_text2 = gr.Textbox(lines=6, label="Enter News Article")
        output2 = gr.Textbox(label="Comparison Result")
        btn2 = gr.Button("Compare Models")
        btn2.click(fn=compare_news, inputs=input_text2, outputs=output2)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://3e98372f63b12a2a19.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
